In [2]:
%pip install -q langgraph==0.2.57 langchain-ibm==0.3.10

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
OPENAI_ORG_ID = os.getenv("OPENAI_ORG_ID", "")

# Model Configuration
DEFAULT_MODEL = os.getenv("DEFAULT_MODEL", "gpt-3.5-turbo")
TEMPERATURE = float(os.getenv("TEMPERATURE", "0.7"))
MAX_TOKENS = int(os.getenv("MAX_TOKENS", "500"))
WATSON_URL = os.getenv("IBM_URL_END_POINT")
PROJECT_ID = os.getenv("IBM_PROJECT_ID")
IBM_API_KEY = os.getenv("IBM_API_KEY")


In [4]:
from langgraph.graph import StateGraph

from langchain_openai import ChatOpenAI
from langchain_ibm import ChatWatsonx
openai_llm = ChatOpenAI(
    model="gpt-4.1-nano",
    api_key = OPENAI_API_KEY,
)
watsonx_llm = ChatWatsonx(
    model_id="ibm/granite-8b-code-instruct",
    url="https://us-south.ml.cloud.ibm.com",
    project_id=PROJECT_ID,
    apikey=IBM_API_KEY,

)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/samuel/Desktop/ai_applications_coursera/venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/samuel/Desktop/ai_applications_coursera/venv/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/home/samuel/Desktop/ai_applications_coursera/venv/lib/python3.12/site-packages/ipyk

In [5]:
from typing import TypedDict, Optional

class AuthState(TypedDict):
    username: Optional[str] 
    password: Optional[str]
    is_authenticated: Optional[bool]
    output: Optional[str]

In [6]:
auth_state_1: AuthState = {
    "username": "alice123",
    "password": "123",
    "is_authenticated": True,
    "output": "Login successful."
}
print(f"auth_state_1: {auth_state_1}")

auth_state_1: {'username': 'alice123', 'password': '123', 'is_authenticated': True, 'output': 'Login successful.'}


In [7]:
auth_state_2: AuthState = {
    "username":"",
    "password": "wrongpassword",
    "is_authenticated": False,
    "output": "Authentication failed. Please try again."
}
print(f"auth_state_2: {auth_state_2}")

auth_state_2: {'username': '', 'password': 'wrongpassword', 'is_authenticated': False, 'output': 'Authentication failed. Please try again.'}


In [8]:
def input_node(state):
    print(state)
    if state.get('username', "") =="":
        username = input("What is your username?")

    password = input("Enter your password: ")
    
    if state.get('username', "") =="":
        return {"username":username, "password":password}
    else:
        return {"password": password}

In [11]:
input_node(auth_state_1)

{'username': 'alice123', 'password': '123', 'is_authenticated': True, 'output': 'Login successful.'}


{'password': '123'}

In [10]:
input_node(auth_state_2)

{'username': '', 'password': 'wrongpassword', 'is_authenticated': False, 'output': 'Authentication failed. Please try again.'}


{'username': '', 'password': ''}

In [12]:
def validate_credentials_node(state):
    # Extract username and password from the state
    username = state.get("username", "")
    password = state.get("password", "")

    print("Username :", username, "Password :", password)
    # Simulated credential validation
    if username == "test_user" and password == "secure_password":
        is_authenticated = True
    else:
        is_authenticated = False

    # Return the updated state with authentication result
    return {"is_authenticated": is_authenticated}

In [13]:
validate_credentials_node(auth_state_1)

Username : alice123 Password : 123


{'is_authenticated': False}

In [14]:
auth_state_3: AuthState = {
    "username":"test_user",
    "password":  "secure_password",
    "is_authenticated": False,
    "output": "Authentication failed. Please try again."
}
print(f"auth_state_3: {auth_state_3}")

auth_state_3: {'username': 'test_user', 'password': 'secure_password', 'is_authenticated': False, 'output': 'Authentication failed. Please try again.'}


In [15]:
validate_credentials_node(auth_state_3)

Username : test_user Password : secure_password


{'is_authenticated': True}

In [16]:
# Define the success node
def success_node(state):
    return {"output": "Authentication successful! Welcome."}

In [17]:
success_node(auth_state_3)

{'output': 'Authentication successful! Welcome.'}

In [18]:
# Define the failure node
def failure_node(state):
    return {"output": "Not Successfull, please try again!"}

In [19]:
def router(state):
    if state['is_authenticated']:
        return "success_node"
    else:
        return "failure_node"

In [20]:
from langgraph.graph import StateGraph
from langgraph.graph import END

# Create an instance of StateGraph with the GraphState structure
workflow = StateGraph(AuthState)
workflow

In [21]:
workflow.add_node("InputNode", input_node)
workflow.add_node("ValidateCredential", validate_credentials_node)
workflow.add_node("Success", success_node)
workflow.add_node("Failure", failure_node)

In [24]:
workflow.add_edge("InputNode", "ValidateCredential")
workflow.add_edge("Success", END)
workflow.add_edge("Failure", "InputNode")
workflow.add_conditional_edges("ValidateCredential", router, {"success_node": "Success", "failure_node": "Failure"})

In [25]:
workflow.set_entry_point("InputNode")
app = workflow.compile()

In [27]:
inputs = {"username": "test_user"}
result = app.invoke(inputs)
print(result)

{'username': 'test_user'}
Username : test_user Password : 123
{'username': 'test_user', 'password': '123', 'is_authenticated': False, 'output': 'Not Successfull, please try again!'}
Username : test_user Password : 123
{'username': 'test_user', 'password': '123', 'is_authenticated': False, 'output': 'Not Successfull, please try again!'}
Username : test_user Password : secure_password
{'username': 'test_user', 'password': 'secure_password', 'is_authenticated': True, 'output': 'Authentication successful! Welcome.'}


In [28]:
result['output']

'Authentication successful! Welcome.'

In [29]:
# Define the structure of the QA state
class QAState(TypedDict):
    # 'question' stores the user's input question. It can be a string or None if not provided.
    question: Optional[str]
    
    # 'context' stores relevant context about the guided project, if the question pertains to it.
    # If the question isn't related to the project, this will be None.
    context: Optional[str]
    
    # 'answer' stores the generated response or answer. It can be None until the answer is generated.
    answer: Optional[str]

In [30]:
# Create an example object
qa_state_example = QAState(
    question="What is the purpose of this guided project?",
    context="This project focuses on building a chatbot using Python.",
    answer=None
)

# Print the attributes
for key, value in qa_state_example.items():
    print(f"{key}: {value}")

question: What is the purpose of this guided project?
context: This project focuses on building a chatbot using Python.
answer: None


In [31]:
def input_validation_node(state):
    # Extract the question from the state, and strip any leading or trailing spaces
    question = state.get("question", "").strip()
    
    # If the question is empty, return an error message indicating invalid input
    if not question:
        return {"valid": False, "error": "Question cannot be empty."}
    
    # If the question is valid, return valid status
    return {"valid": True}

In [32]:
input_validation_node(qa_state_example)

{'valid': True}

In [33]:
def context_provider_node(state):
    question = state.get("question", "").lower()
    # Check if the question is related to the guided project
    if "langgraph" in question or "guided project" in question:
        context = (
            "This guided project is about using LangGraph, a Python library to design state-based workflows. "
            "LangGraph simplifies building complex applications by connecting modular nodes with conditional edges."
        )
        return {"context": context}
    # If unrelated, set context to null
    return {"context": None}

In [35]:
from langchain_ibm import ChatWatsonx

llm = ChatWatsonx(
    model_id="meta-llama/llama-4-maverick-17b-128e-instruct-fp8",
    url="https://us-south.ml.cloud.ibm.com",
    project_id=PROJECT_ID,
    apikey=IBM_API_KEY,
)

In [36]:
def llm_qa_node(state):
    # Extract the question and context from the state
    question = state.get("question", "")
    context = state.get("context", None)

    # Check for missing context and return a fallback response
    if not context:
        return {"answer": "I don't have enough context to answer your question."}

    # Construct the prompt dynamically
    prompt = f"Context: {context}\nQuestion: {question}\nAnswer the question based on the provided context."

    # Use LangChain's ChatOpenAI to get the response
    try:
        response = llm.invoke(prompt)
        return {"answer": response.content.strip()}
    except Exception as e:
        return {"answer": f"An error occurred: {str(e)}"}

In [37]:
qa_workflow = StateGraph(QAState)
qa_workflow.add_node("InputNode", input_validation_node)
qa_workflow.add_node("ContextNode", context_provider_node)
qa_workflow.add_node("QANode", llm_qa_node)

In [38]:
qa_workflow.set_entry_point("InputNode")

In [39]:
qa_workflow.add_edge("InputNode", "ContextNode")
qa_workflow.add_edge("ContextNode", "QANode")
qa_workflow.add_edge("QANode", END)

In [40]:
qa_app = qa_workflow.compile()

In [41]:
qa_app.invoke({"question": "What is the weather today?"})

{'question': 'What is the weather today?',
 'context': None,
 'answer': "I don't have enough context to answer your question."}

In [42]:
qa_app.invoke({"question": "What is LangGraph?"})

{'question': 'What is LangGraph?',
 'context': 'This guided project is about using LangGraph, a Python library to design state-based workflows. LangGraph simplifies building complex applications by connecting modular nodes with conditional edges.',
 'answer': 'LangGraph is a Python library used to design state-based workflows. It simplifies building complex applications by connecting modular nodes with conditional edges.'}

In [43]:
qa_app.invoke({"question": "What is the best guided project?"})

{'question': 'What is the best guided project?',
 'context': 'This guided project is about using LangGraph, a Python library to design state-based workflows. LangGraph simplifies building complex applications by connecting modular nodes with conditional edges.',
 'answer': 'Based on the context provided, the best guided project is the one about using LangGraph, a Python library to design state-based workflows. This is because LangGraph simplifies building complex applications by connecting modular nodes with conditional edges, making it a valuable tool for various applications.'}

In [44]:
import random
import string
from typing import TypedDict

from langgraph.graph import StateGraph, END

In [45]:
class ChainState(TypedDict):
    n: int
    letter: str

In [46]:
def add(state: ChainState) -> ChainState:
    random_letter = random.choice(string.ascii_letters)
    return {
        **state,
        "n": state["n"] + 1,
        "letter": random_letter
    }

In [47]:
def print_out(state: ChainState) -> ChainState:
    print("Current n:", state["n"], "Letter", state["letter"])
    return state

In [48]:
def stop_condition(state: ChainState) -> bool:
    return state["n"] >= 13

In [49]:
workflow = StateGraph(ChainState)

workflow.add_node("add", add)
workflow.add_node("print", print)
workflow.add_edge("add", "print")
workflow.add_conditional_edges("print", stop_condition, {True: END, False: "add"})
workflow.set_entry_point("add")

In [50]:
app = workflow.compile()
result = app.invoke({"n":1, "letter": ""})

{'n': 2, 'letter': 'K'}
{'n': 3, 'letter': 'k'}
{'n': 4, 'letter': 'k'}
{'n': 5, 'letter': 'J'}
{'n': 6, 'letter': 'A'}
{'n': 7, 'letter': 'L'}
{'n': 8, 'letter': 'H'}
{'n': 9, 'letter': 's'}
{'n': 10, 'letter': 'T'}
{'n': 11, 'letter': 'U'}
{'n': 12, 'letter': 'i'}
{'n': 13, 'letter': 'F'}
